# Intermediate representations

What the learned adapters do to the raw multiplexed stack. One row per example crop: DAPI, Max-Projection, then for each of
the 4 trainable models the output of its 1x1 adapter and of its tuned ChannelNet AdaptorNet (both shown as RGB).
Runs the trained checkpoints under `training/*/Experiments/`, which are not part of the repository.

In [ ]:
import sys
from functools import lru_cache
from pathlib import Path

BENCH_DIR = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if p.name == "bench")
sys.path.insert(0, str(BENCH_DIR))

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch

import figures.render_utils as ru
from training.common import channels as ch
from validation.consolidate import DATASETS

matplotlib.rcParams["pdf.fonttype"] = 42
EXPORT_DIR = BENCH_DIR / "figures" / "export" / "intermediate"
TRAINING = BENCH_DIR / "training"

# Per model: training subdirectory, Stage-2 pretrained checkpoint, run-name prefix, and the folder and file
# extension its 1x1 and ChannelNet-tuned checkpoints were saved under (under Experiments/{dataset}/).
MODELS = {
    "Dist U-Net": dict(dir="UNet", pretrained="Stage_2/DIST_STAGE2.ckpt", prefix="DIST",
                       tuned_dir="models", x1_dir="models", x1_ext=".ckpt"),
    "Stardist": dict(dir="CellViT_Star", pretrained="Stage_2/checkpoints/Stardist_STAGE2_BEST.pt", prefix="Stardist",
                     tuned_dir="saves", x1_dir="saves", x1_ext=".pt"),
    "Cellpose": dict(dir="Cellpose", pretrained="Stage_2/CPose2ch_STAGE2_full_BEST.pt", prefix="CPose2ch",
                     tuned_dir="runs", x1_dir="runs", x1_ext=".pt"),
    "InstanSeg": dict(dir="InstanSeg", pretrained="Stage_2/checkpoints/InstanSeg_STAGE2_BEST.pt", prefix="InstanSeg",
                      tuned_dir="saves", x1_dir="models", x1_ext=".pt"),
}
FOLDS = {"Vectra": 0, "CODEX": 0, "MACSima": 0, "Zeiss": 4}  # fold whose trained checkpoints are used, per dataset

## Load the trained adapters and project

In [ ]:
@lru_cache(maxsize=None)
def load_tuned(model, dataset, fold):
    """Pretrained segmenter with a fresh AdaptorNet, then the trained ChannelNet-tuned weights."""
    spec = MODELS[model]
    pretrained = str(TRAINING / spec["dir"] / spec["pretrained"])
    if model == "Dist U-Net":
        from modelling.UNet.AdaptorDistNet import build
        net = build(pretrained)
    elif model == "Stardist":
        from modelling.Cellvit.AdaptorStarDistNet import build
        net = build(pretrained, device="cpu")
    elif model == "Cellpose":
        from modelling.Cellpose.AdaptorCellposeNet import build
        net = build(pretrained, gpu=False)
    elif model == "InstanSeg":
        from modelling.InstanSeg.AdaptorInstanSegNet import build
        net = build(pretrained, device="cpu")
    ckpt = TRAINING / spec["dir"] / "Experiments" / dataset / spec["tuned_dir"] / \
        f"{spec['prefix']}_{dataset}_ChannelNetTuned_filtered_fold{fold}_BEST.pt"
    net.load_state_dict(torch.load(ckpt, map_location="cpu"))
    return net.eval()


@lru_cache(maxsize=None)
def load_1x1(model, dataset, fold):
    """Pretrained segmenter with the two 1x1 convs prepended, then the trained 1x1 weights."""
    spec = MODELS[model]
    pretrained = str(TRAINING / spec["dir"] / spec["pretrained"])
    net = ru.RENDER_1X1_SPECS[model]["build"](pretrained, ch.n_input_channels(dataset, "selected", "1x1"))

    folder = TRAINING / spec["dir"] / "Experiments" / dataset / spec["x1_dir"]
    for tag in ("_selected", "_filtered"):
        ckpt = folder / f"{spec['prefix']}_{dataset}_1x1{tag}_fold{fold}_full_BEST{spec['x1_ext']}"
        if ckpt.exists():
            break
    state = torch.load(ckpt, map_location="cpu")
    state = state["state_dict"] if "state_dict" in state else state
    # each model's training script saved a different part of the network
    target = net.net if model == "Cellpose" else net.backbone if model == "InstanSeg" else net
    target.load_state_dict(state)
    target.eval()
    return net


def normalize(x):
    """Per-channel min-max to [0, 1]."""
    lo, hi = x.min(axis=(0, 1)), x.max(axis=(0, 1))
    return (x - lo) / np.where(hi - lo == 0, 1, hi - lo)


def to_tensor(x):
    return torch.from_numpy(x.astype(np.float32)).permute(2, 0, 1).unsqueeze(0)


def project_tuned(raw, model, dataset, fold):
    """AdaptorNet's 3-channel projection of `raw` (H, W, C); padded to a multiple of 32 for the network."""
    h, w = raw.shape[:2]
    padded = np.pad(raw.astype(np.float32), ((0, -h % 32), (0, -w % 32), (0, 0)))
    with torch.no_grad():
        out = load_tuned(model, dataset, fold).project(to_tensor(normalize(padded)))
    return normalize(out[0].permute(1, 2, 0).numpy()[:h, :w])


def project_1x1(raw, model, dataset, fold):
    """Output of just the two 1x1 conv layers, not the rest of the backbone."""
    layer1, layer2 = ru._1x1_layers(load_1x1(model, dataset, fold), model)
    with torch.no_grad():
        out = layer2(layer1(to_tensor(normalize(raw.astype(np.float32)))))
    return normalize(out[0].permute(1, 2, 0).numpy())


def to_rgb(img):
    """Pad Cellpose's 2-channel projection with a zero channel for display."""
    return np.concatenate([img, np.zeros_like(img[..., :1])], axis=-1) if img.shape[-1] == 2 else img[..., :3]

## One row per crop

In [ ]:
def plot_row(ex, dataset, suptitle):
    fold = FOLDS[dataset]
    panels = [("DAPI", ex["raw"][:, :, 0], "gray"), ("Max-Proj", ex["max_proj"], None)]
    for model in MODELS:
        panels.append((f"1x1\n({model})", to_rgb(project_1x1(ex["raw"], model, dataset, fold)), None))
        panels.append((f"ChannelNet-tuned\n({model})", to_rgb(project_tuned(ex["raw"], model, dataset, fold)), None))

    fig, axs = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3.4))
    for ax, (title, img, cmap) in zip(axs, panels):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=11)
        ax.axis("off")
    fig.suptitle(suptitle, fontsize=15, y=1.05)
    fig.tight_layout()
    return fig

### One example per dataset

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for dataset in DATASETS:
    fig = plot_row(ru.load_example(dataset), dataset, dataset)
    fig.savefig(EXPORT_DIR / f"{dataset}_intermediate_row.pdf", bbox_inches="tight")
    plt.close(fig)

### Gallery: every curated example (`render_utils.GALLERY_EXAMPLES`)

In [ ]:
(EXPORT_DIR / "gallery").mkdir(parents=True, exist_ok=True)
for dataset in DATASETS:
    for i, ex in enumerate(ru.load_gallery_examples(dataset)):
        stem = Path(ex["label_path"]).stem
        fig = plot_row(ex, dataset, f"{dataset} -- gallery {i} ({stem})")
        fig.savefig(EXPORT_DIR / "gallery" / f"{dataset}_gallery{i}_intermediate_row.pdf", bbox_inches="tight", dpi=300)
        plt.close(fig)
        print(f"saved {dataset} gallery {i}")